In [2]:
!pip install wandb
!pip install optuna
!pip install optuna-integration[wandb]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 kB 4.8 MB/s eta 0:00:00


In [3]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
my_secret = user_secrets.get_secret("wandb_api_key")
wandb.login(key=my_secret)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: midonik10 (midonik10-wsb-merito) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
# ========================================
# 1. Importy i ustawienia ogólne
# ========================================
import warnings
warnings.filterwarnings("ignore")
import os
import tensorflow as tf
import optuna
import wandb
from optuna.integration.wandb import WeightsAndBiasesCallback
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing import image_dataset_from_directory

print("TensorFlow:", tf.__version__)

2025-04-29 19:24:23.320097: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745954663.564200      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745954663.631867      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.0


In [5]:
# ========================================
# 2. Wymuszenie i optymalizacja użycia GPU
# ========================================
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU dostępne i gotowe:", gpus)
    except RuntimeError as e:
        print("Błąd przy ustawianiu GPU:", e)
else:
    print("Brak GPU, będzie użyty CPU")

GPU dostępne i gotowe: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [6]:
# ========================================
# 3. Stałe i konfiguracja ścieżek
# ========================================
TRAIN_DIR = "/kaggle/input/plant-village-balanced/train"
TEST_DIR  = "/kaggle/input/plant-village-balanced/test"
NUM_CLASSES = len(next(os.walk(TRAIN_DIR))[1])

study_name = "alexnet_optuna_v1"
storage_url = f"sqlite:///{study_name}.db"

IMG_SIZE = (224, 224)
IMG_SHAPE = IMG_SIZE + (3,)
SEED = 42

wandb_kwargs = {
    "project": "alexnet-optuna-wandb",
    "group":   "alexnet-optuna-gpu",
    "reinit":  True
}
wandb.init(**wandb_kwargs)

wandbc = WeightsAndBiasesCallback(
    wandb_kwargs=wandb_kwargs,
    metric_name="val_accuracy",
    as_multirun=True
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


In [7]:
# ========================================
# 4. Przetwarzanie danych
# ========================================
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def build_datasets(batch_size):
    train_ds = image_dataset_from_directory(
        TRAIN_DIR,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=batch_size
    ).map(preprocess).prefetch(tf.data.AUTOTUNE)

    val_ds = image_dataset_from_directory(
        TRAIN_DIR,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=batch_size
    ).map(preprocess).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds

In [8]:
# ========================================
# 5. Budowa modelu AlexNet
# ========================================
def build_alexnet_model(input_shape=(224, 224, 3), num_classes=1000, dropout_rate=0.5, dense_units=128):
    inputs = Input(shape=input_shape)

    x = Conv2D(96, (11, 11), strides=(4, 4), activation='relu', padding='valid')(inputs)
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2))(x)

    x = Conv2D(256, (5, 5), padding='same', activation='relu')(x)
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2))(x)

    x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2))(x)

    x = Flatten()(x)
    x = Dense(4096, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(4096, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

In [9]:
# ========================================
# 6. Funkcja celu Optuna
# ========================================
def objective(trial):
    K.clear_session()

    lr = trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True)
    dr = trial.suggest_float("dropout", 0.4, 0.5)
    du = trial.suggest_categorical("dense_units", [128])
    bs = trial.suggest_categorical("batch_size", [8, 16])

    train_ds, val_ds = build_datasets(bs)

    model = build_alexnet_model(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES, dropout_rate=dr, dense_units=du)

    optimizer = Adam(learning_rate=lr, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15,
        callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
        verbose=1
    )

    return max(history.history["val_accuracy"])

In [ ]:
# ========================================
# 7. Uruchomienie Optuny
# ========================================
study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_url,
    load_if_exists=True
)
study.optimize(objective, n_trials=30, callbacks=[wandbc])

print("Najlepsze parametry:")
print(study.best_params)

In [13]:
import wandb
import json

# Upewnij się, że jesteś zalogowany
wandb.login()

# Połącz się z konkretnym runem
api = wandb.Api()
run = api.run("midonik10-wsb-merito/alexnet-optuna-wandb/trial-1_hearty-capybara-2")

# Zapisz konfigurację (hiperparametry)
with open("run_config.json", "w") as f:
    json.dump(dict(run.config), f, indent=2)

# Zapisz podsumowanie metryk (np. val_accuracy, loss, accuracy itd.)
with open("run_summary.json", "w") as f:
    json.dump(dict(run.summary), f, indent=2)

print("Zapisano: run_config.json i run_summary.json")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


CommError: Could not find run <Run midonik10-wsb-merito/alexnet-optuna-wandb/trial-1_hearty-capybara-2 (not found)>

In [10]:
# ========================================
# 8. Trening finalny najlepszego modelu
# ========================================
def build_final_model(params, num_classes):
    model = build_alexnet_model(
        input_shape=IMG_SHAPE,
        num_classes=num_classes,
        dropout_rate=params["dropout"],
        dense_units=params["dense_units"]
    )
    model.compile(
        optimizer=Adam(learning_rate=params["learning_rate"]),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

train_ds_final, val_ds_final = build_datasets(batch_size=32)
num_classes = 39  # lub dynamicznie jak wyżej

best_model = build_final_model(study.best_params, num_classes)

best_model.fit(
    train_ds_final,
    validation_data=val_ds_final,
    epochs=30,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)

best_model.save("best_alexnet_model.keras")
print("Model zapisany jako best_alexnet_model.keras")

Found 62400 files belonging to 39 classes.
Using 49920 files for training.


I0000 00:00:1745955921.415413      31 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1745955921.419001      31 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1745955921.419505      31 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1745955921.419874      31 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 62400 files belonging to 39 classes.
Using 12480 files for validation.


NameError: name 'study' is not defined